In [0]:
# Netflix dataset using PySpark

df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/netflix_data/netflix_titles.csv")


display(df.limit(5))

show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,null,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable."
s2,TV Show,Blood & Water,null,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth."
s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noureddine Farihi, Geert Van Rampelberg, Bakary Diombera",null,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adventure","To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war."
s4,TV Show,Jailbirds New Orleans,null,null,null,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down among the incarcerated women at the Orleans Justice Center in New Orleans on this gritty reality series."
s5,TV Show,Kota Factory,null,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam Khan, Ahsaas Channa, Revathi Pillai, Urvi Singh, Arun Kumar",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV Comedies","In a city of coaching centers known to train India’s finest collegiate minds, an earnest but unexceptional student and his friends navigate campus life."


In [0]:
# Display the dataset schema

df.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [0]:
# Count Movies vs TV Shows

df.groupBy("type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+-------------+-----+
|         type|count|
+-------------+-----+
|        Movie| 6131|
|      TV Show| 2676|
|         NULL|    1|
|William Wyler|    1|
+-------------+-----+



In [0]:
# Count NULL / Missing Director values

from pyspark.sql import functions as F

director_null_count = df.filter(
    F.col("director").isNull()
).count()

director_empty_count = df.filter(
    F.col("director").isNotNull() &
    (F.trim(F.col("director")) == "")
).count()

total_missing_directors = director_null_count + director_empty_count

print("NULL directors:", director_null_count)
print("Empty directors:", director_empty_count)
print("Total missing directors:", total_missing_directors)

NULL directors: 2636
Empty directors: 0
Total missing directors: 2636


In [0]:
# Exercise 3: Count titles per rating

rating_counts = df.groupBy("rating") \
    .count() \
    .orderBy("count", ascending=False)

display(rating_counts)

rating,count
TV-MA,3195
TV-14,2158
TV-PG,862
R,796
PG-13,489
TV-Y7,334
TV-Y,307
PG,286
TV-G,220
NR,80


In [0]:
# Exercise 4: Top 10 genres by title count

from pyspark.sql.functions import col, split, explode, trim, countDistinct

genre_counts = df.select(
    col("show_id"),
    explode(split(col("listed_in"), ",")).alias("genre")
)

genre_counts = genre_counts.withColumn(
    "genre",
    trim(col("genre"))
)

top_10_genres = genre_counts \
    .filter(col("genre").isNotNull() & (col("genre") != "")) \
    .groupBy("genre") \
    .agg(countDistinct("show_id").alias("title_count")) \
    .orderBy("title_count", ascending=False) \
    .limit(10)

display(top_10_genres)

genre,title_count
International Movies,2748
Dramas,2419
Comedies,1670
International TV Shows,1350
Documentaries,866
Action & Adventure,857
TV Dramas,762
Independent Movies,751
Children & Family Movies,641
Romantic Movies,616


In [0]:
# Exercise 5: Top 15 genres by title count

top_15_genres = genre_counts \
    .filter(col("genre").isNotNull() & (col("genre") != "")) \
    .groupBy("genre") \
    .agg(countDistinct("show_id").alias("title_count")) \
    .orderBy("title_count", ascending=False) \
    .limit(15)

display(top_15_genres)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8151311625512032>, line 3
      1 # Exercise 5: Top 15 genres by title count
----> 3 top_15_genres = genre_counts \
      4     .filter(col("genre").isNotNull() & (col("genre") != "")) \
      5     .groupBy("genre") \
      6     .agg(countDistinct("show_id").alias("title_count")) \
      7     .orderBy("title_count", ascending=False) \
      8     .limit(15)
     10 display(top_15_genres)

NameError: name 'genre_counts' is not defined

In [0]:
# Exercise 6: Type of Available Content

from pyspark.sql.functions import (
    col,
    count,
    round,
    sum as spark_sum
)

from pyspark.sql.window import Window

# Keep only valid content types
valid_content = df.filter(
    col("type").isin("Movie", "TV Show")
)

# Count titles by content type
content_type_counts = valid_content \
    .groupBy("type") \
    .agg(count("*").alias("total_titles"))

# Define window specification
window_spec = Window.partitionBy()

# Calculate percentage
content_type_analysis = content_type_counts \
    .withColumn(
        "percentage",
        round(
            col("total_titles") /
            spark_sum("total_titles").over(window_spec) * 100,
            2
        )
    ) \
    .orderBy("total_titles", ascending=False)

display(content_type_analysis)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


type,total_titles,percentage
Movie,6131,69.62
TV Show,2676,30.38


In [0]:

# Reload Netflix dataset

df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/netflix_data/netflix_titles.csv")

# Verify dataset
display(df.limit(5))


show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,null,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable."
s2,TV Show,Blood & Water,null,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth."
s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noureddine Farihi, Geert Van Rampelberg, Bakary Diombera",null,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adventure","To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war."
s4,TV Show,Jailbirds New Orleans,null,null,null,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down among the incarcerated women at the Orleans Justice Center in New Orleans on this gritty reality series."
s5,TV Show,Kota Factory,null,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam Khan, Ahsaas Channa, Revathi Pillai, Urvi Singh, Arun Kumar",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV Comedies","In a city of coaching centers known to train India’s finest collegiate minds, an earnest but unexceptional student and his friends navigate campus life."


In [0]:
# Investigate invalid DATE_ADDED values

from pyspark.sql.functions import (
    col,
    trim,
    try_to_timestamp,
    lit
)

df_checked = df.withColumn(
    "parsed_date",
    try_to_timestamp(
        trim(col("date_added")),
        lit("MMMM d, yyyy")
    )
)

invalid_dates = df_checked.filter(
    col("date_added").isNotNull() &
    col("parsed_date").isNull()
)

display(
    invalid_dates.select(
        "show_id",
        "type",
        "title",
        "date_added",
        "rating",
        "duration"
    )
)

show_id,type,title,date_added,rating,duration
s74,Movie,King of Boys,Toni Tones,Jide Kosoko,"Sharon Ooja"""
s171,Movie,Rhyme & Reason,Desiree Densiti,E-40,MC Eiht
s199,TV Show,King of Boys: The Return of the King,"Charles """"Charly Boy"""" Oputa",Keppy Ekpenyong Bassey,Bimbo Manuel
s1252,Movie,Dances with Wolves,Robert Pastorelli,Maury Chaykin,"Jimmy Herman"""
s1412,Movie,What Would Sophia Loren Do?,United States,2021,TV-14
s1416,Movie,An Imperfect Murder,United States,2017,R
s1549,TV Show,The Challenge,United States,2006,TV-14
s1769,Movie,The Next Karate Kid,"Walton Goggins""","November 1, 2020",1994
s2138,Movie,An Easy Girl,France,2019,TV-MA
s2211,Movie,Offering to the Storm,Imanol Arias,Itziar Aizpuru,Benn Northover


In [0]:
# Exercise 7: Newly added contents by month and year

from pyspark.sql.functions import (
    col,
    trim,
    try_to_timestamp,
    lit,
    year,
    month,
    countDistinct
)

# Parse valid dates
df_dates = df.withColumn(
    "added_date",
    try_to_timestamp(
        trim(col("date_added")),
        lit("MMMM d, yyyy")
    )
)

# Extract year and month
df_dates = df_dates \
    .withColumn("year", year(col("added_date"))) \
    .withColumn("month", month(col("added_date")))

# Count titles by month and year
monthly_additions = df_dates \
    .filter(col("added_date").isNotNull()) \
    .groupBy("year", "month") \
    .agg(
        countDistinct("show_id").alias("total_titles")
    ) \
    .orderBy("year", "month")

display(monthly_additions)

year,month,total_titles
2008,1,1
2008,2,1
2009,5,1
2009,11,1
2010,11,1
2011,5,1
2011,9,1
2011,10,11
2012,2,1
2012,11,1


In [0]:
# Top 10 months with the most added titles

top_10_months = monthly_additions \
    .orderBy("total_titles", ascending=False) \
    .limit(10)

display(top_10_months)

year,month,total_titles
2021,7,257
2019,11,255
2019,12,215
2021,6,206
2020,1,204
2019,10,193
2018,10,191
2021,4,188
2018,12,185
2021,9,181


In [0]:
# Exercise 7: Total titles added per year

from pyspark.sql.functions import sum as spark_sum

yearly_additions = monthly_additions \
    .groupBy("year") \
    .agg(
        spark_sum("total_titles").alias("total_titles")
    ) \
    .orderBy("total_titles", ascending=False)

display(yearly_additions)

year,total_titles
2019,2014
2020,1873
2018,1647
2021,1491
2017,1186
2016,429
2015,81
2014,24
2011,13
2013,11


In [0]:

# Exercise 8: Average duration of TV Shows in seasons

from pyspark.sql.functions import (
    col,
    regexp_extract,
    avg,
    round,
    count,
    when
)

# Filter TV Shows
tv_shows = df.filter(
    col("type") == "TV Show"
)

# Extract number of seasons safely
season_number = regexp_extract(
    col("duration"),
    r"^(\d+)\s+Seasons?$",
    1
)

tv_shows = tv_shows.withColumn(
    "seasons",
    when(
        season_number != "",
        season_number.cast("int")
    )
)

# Calculate average seasons
average_seasons = tv_shows \
    .filter(col("seasons").isNotNull()) \
    .agg(
        round(avg("seasons"), 2).alias("average_seasons"),
        count("seasons").alias("total_tv_shows")
    )

display(average_seasons)


average_seasons,total_tv_shows
1.77,2673


In [0]:

# Exercise 9: Christmas Movies and TV Shows

from pyspark.sql.functions import (
    col,
    lower,
    concat_ws
)

# Search for Christmas-related content
christmas_content = df.filter(
    lower(
        concat_ws(
            " ",
            col("title"),
            col("description"),
            col("listed_in")
        )
    ).contains("christmas")
)

# Select relevant columns
christmas_titles = christmas_content.select(
    "show_id",
    "type",
    "title",
    "release_year",
    "rating",
    "listed_in",
    "description"
).orderBy("release_year", ascending=False)

display(christmas_titles)


show_id,type,title,release_year,rating,listed_in,description
s1682,Movie,The Princess Switch: Switched Again,2020,TV-G,"Children & Family Movies, Comedies, Dramas","When Margaret's Christmas coronation complicates her love life, her double Stacy steps in to save the day. But will a third look-alike ruin their plan?"
s1636,Movie,A Go! Go! Cory Carson Christmas,2020,TV-Y,Children & Family Movies,"When a familiar-looking stranger crashes in without a memory, Cory helps him remember the magic of Christmas to save the holiday for everyone."
s1667,Movie,Alien Xmas,2020,TV-Y,Children & Family Movies,"A young elf mistakes a tiny alien for a Christmas gift, not knowing her new plaything has plans to destroy Earth's gravity — and steal all the presents."
s1592,Movie,Captain Underpants Mega Blissmas,2020,TV-Y7,"Children & Family Movies, Comedies",Christmas gets weird — really weird — after George and Harold go back in time to change up a few of their beloved holiday's traditions.
s1536,TV Show,How To Ruin Christmas,2020,TV-MA,"International TV Shows, TV Comedies, TV Dramas",Prodigal daughter Tumi tries to make things right after completely ruining what should have been her sister's picture-perfect Christmas wedding.
s1685,TV Show,Holiday Home Makeover with Mr. Christmas,2020,TV-G,Reality TV,"""Interior designer Benjamin “Mr. Christmas” Bradley works with a trusty team of """"elves” to help families transform their homes for the holidays."""
s1642,TV Show,Sugar Rush Christmas,2020,TV-14,Reality TV,"""It's everything you love about """"Sugar Rush"""" – with a holly jolly holiday twist – in this Christmas-themed spin on competitive baking."""
s1628,TV Show,The Holiday Movies That Made Us,2020,TV-PG,Docuseries,"Unwrap the real stories behind these iconic Christmas blockbusters, thanks to insider interviews and behind-the-scenes peeks."
s2865,Movie,Velvet Colección: Grand Finale,2020,TV-14,"Dramas, International Movies, Romantic Movies","During Christmas 1969, the impending sale of Velvet reunites past and present staff in Madrid for fond remembrances, second chances and new beginnings."
s1652,Movie,The Christmas Chronicles: Part Two,2020,PG,"Children & Family Movies, Comedies","Unhappy over her mom’s new relationship, a now-teenage Kate runs away and lands at the North Pole, where a naughty elf is plotting to cancel Christmas."


In [0]:

# Reload Netflix dataset

df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/netflix_data/netflix_titles.csv")

# Verify dataset
display(df.limit(5))


show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,null,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable."
s2,TV Show,Blood & Water,null,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth."
s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noureddine Farihi, Geert Van Rampelberg, Bakary Diombera",null,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adventure","To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war."
s4,TV Show,Jailbirds New Orleans,null,null,null,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down among the incarcerated women at the Orleans Justice Center in New Orleans on this gritty reality series."
s5,TV Show,Kota Factory,null,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam Khan, Ahsaas Channa, Revathi Pillai, Urvi Singh, Arun Kumar",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV Comedies","In a city of coaching centers known to train India’s finest collegiate minds, an earnest but unexceptional student and his friends navigate campus life."


In [0]:

# Exercise 10: Ryan Reynolds Movies and TV Shows

from pyspark.sql.functions import col, lower

# Find titles featuring Ryan Reynolds
ryan_reynolds_content = df.filter(
    lower(col("cast")).contains("ryan reynolds")
)

# Select relevant columns
ryan_reynolds_titles = ryan_reynolds_content.select(
    "show_id",
    "type",
    "title",
    "release_year",
    "rating",
    "duration",
    "listed_in"
).orderBy("release_year", ascending=False)

# Display results
display(ryan_reynolds_titles)


show_id,type,title,release_year,rating,duration,listed_in
s3151,Movie,6 Underground,2019,R,129 min,"Action & Adventure, Dramas"
s3032,Movie,Betty White: First Lady of Television,2018,TV-14,56 min,Documentaries
s7475,Movie,Mississippi Grind,2015,R,109 min,"Dramas, Independent Movies"
s3583,Movie,Selfless,2015,PG-13,117 min,"Sci-Fi & Fantasy, Thrillers"
s8237,Movie,The Captive,2014,R,112 min,"Dramas, Thrillers"
s8259,Movie,The Croods,2013,PG,99 min,"Children & Family Movies, Comedies"
s8642,Movie,Turbo,2013,PG,96 min,"Children & Family Movies, Comedies"
s47,Movie,Safe House,2012,R,115 min,Action & Adventure
s144,Movie,Green Lantern,2011,PG-13,114 min,"Action & Adventure, Sci-Fi & Fantasy"
s8189,Movie,The Amityville Horror,2005,R,89 min,Horror Movies


In [0]:

# Exercise 11: Movies and TV Shows released during COVID season

from pyspark.sql.functions import col, count, expr

# Convert release_year safely into an integer
covid_df = df.withColumn(
    "clean_release_year",
    expr("try_cast(release_year AS INT)")
)

# Filter valid release years during COVID (2020-2021)
covid_content = covid_df.filter(
    col("clean_release_year").between(2020, 2021)
)

# Count titles by content type
covid_counts = covid_content \
    .groupBy("type") \
    .agg(count("*").alias("total_titles")) \
    .orderBy("total_titles", ascending=False)

# Display results
display(covid_counts)

# Calculate total titles
print("Total titles released during COVID:", covid_content.count())


type,total_titles
Movie,791
TV Show,750


Total titles released during COVID: 1541


In [0]:

# Exercise 12: Movies for a 2-hour flight

from pyspark.sql.functions import col, expr

# Filter movies and extract duration in minutes
flight_movies = df \
    .filter(col("type") == "Movie") \
    .withColumn(
        "duration_minutes",
        expr("try_cast(regexp_extract(duration, '(\\\\d+)\\\\s*min', 1) AS INT)")
    )

# Select movies between 90 and 120 minutes
recommended_movies = flight_movies \
    .filter(col("duration_minutes").between(90, 120)) \
    .select(
        "title",
        "release_year",
        "rating",
        "duration",
        "duration_minutes",
        "listed_in",
        "description"
    ) \
    .orderBy("duration_minutes", ascending=True) \
    .limit(20)

# Display recommendations
display(recommended_movies)


title,release_year,rating,duration,duration_minutes,listed_in,description
Seaspiracy,2021,TV-14,90 min,90,Documentaries,"Passionate about ocean life, a filmmaker sets out to document the harm that humans do to marine species — and uncovers alarming global corruption."
Flipped,2010,PG,90 min,90,"Comedies, Dramas, Romantic Movies","Bryce and Juli first meet as children, with Juli having a crush on Bryce. As they mature, it appears their bond may blossom into something more."
The Old Ways,2020,TV-MA,90 min,90,Horror Movies,"A reporter visits her birthplace in Veracruz for a story about tribal culture, only to be kidnapped by locals who believe she's demonically possessed."
My Amanda,2021,TV-14,90 min,90,"Dramas, International Movies, Romantic Movies","Two unusually close friends share every aspect of their lives together. As their lives evolve, their bond remains the only constant."
Show Dogs,2018,PG,90 min,90,"Children & Family Movies, Comedies",A rough and tough police dog must go undercover with an FBI agent as a prim and proper pet at a dog show to save a baby panda from an illegal sale.
A Champion Heart,2018,G,90 min,90,"Children & Family Movies, Dramas","When a grieving teen must work off her debt to a ranch, she cares for a wounded horse that teaches her more about healing than she expected."
Boyka: Undisputed,2016,R,90 min,90,Action & Adventure,The fourth film in the action franchise finds Boyka taking on brutal gangsters to protect the widow of a man he unintentionally killed in the ring.
Wannabe Courageous,2019,TV-14,90 min,90,"Comedies, International Movies","When his newlywed wife becomes disenchanted because he failed to protect her from danger, Omar seeks the help of a quirky psychiatrist to win her back."
Hospital,2020,TV-MA,90 min,90,"Horror Movies, International Movies","In an abandoned hospital in Tainan, visitors seeking to communicate with their relatives' spirits are haunted by disturbing, supernatural occurrences."
Brick Mansions,2014,PG-13,90 min,90,"Action & Adventure, International Movies",An undercover police detective partners with an ex-convict to take down a drug kingpin amid grime and crime in a walled-off Detroit housing project.
